In [ ]:
# --- Repo-root bootstrap: keep relative paths (data/..., common.validation) working
# regardless of where Jupyter's working directory starts out ---
import os, sys
from pathlib import Path

_root = Path.cwd()
while not (_root / ".git").exists() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/global_excel_resampled_scaled.csv", sep=";")
# ==========================================
# 1. FILTRAGE INITIAL ET SÉLECTION DES PHASES
# ==========================================

# Définition des phases requises et de leur ordre strict
phases_cibles = ["PORT", "ART", "TARD"]

# On ne garde temporairement que les lignes qui concernent ces 3 phases
df = df[df["temps_inj"].isin(phases_cibles)]

# ==========================================
# 2. IDENTIFICATION DES PATIENTS ÉLIGIBLES
# ==========================================

# Un patient est valide s'il possède exactement les 3 phases distinctes
phases_par_patient = df.groupby("id")["temps_inj"].nunique()
patients_valides = phases_par_patient[phases_par_patient == 3].index

# Filtrage du DataFrame pour ne garder que ces patients complets
df = df[df["id"].isin(patients_valides)]

# Transforme la colonne en type Categorical pour imposer l'ordre de tri des colonnes plus tard
df["temps_inj"] = pd.Categorical(
    df["temps_inj"], categories=phases_cibles, ordered=True
)

# ==========================================
# 3. PIVOTEMENT ET RESTRUCTURATION EN WIDE
# ==========================================

# Identification des colonnes de métadonnées à garder comme index de ligne
metadata_cols = ["id", "patient_num", "classe_name"]

# Identification automatique de toutes les colonnes de features radiomiques
feature_cols = [
    col
    for col in df.columns
    if col not in metadata_cols + ["temps_inj", "Unnamed: 0"]
]

# Pivotement du DataFrame
df_wide = df.pivot(
    index=metadata_cols, columns="temps_inj", values=feature_cols
)

# Réorganisation des niveaux d'index des colonnes : on bascule la phase en premier niveau
df_wide = df_wide.swaplevel(0, 1, axis=1)

# Tri des colonnes par le niveau 0 (les phases), garantissant l'ordre : PORT -> ART -> TARD
# sort_remaining=False permet de conserver l'ordre d'origine des features radiomiques
df_wide = df_wide.sort_index(axis=1, level=0, sort_remaining=False)

# ==========================================
# 4. APLATISSEMENT ET NETTOYAGE FINAL
# ==========================================

# Reconstruction des noms de colonnes sous la forme : nom_feature_PHASE
df_wide.columns = [f"{feat}_{phase}" for phase, feat in df_wide.columns]

# Réinitialisation de l'index pour repasser 'id', 'patient_num' et 'classe_name' en colonnes normales
df_final = df_wide.reset_index()

df_final["id"] = df_final["id"].astype(int)

df_clinique = pd.read_csv("./data/Descriptif_patients(Sheet1).csv", sep=";")

# ==========================================
# 5. FUSION, ENCODAGE ET NORMALISATION CLINIQUE
# ==========================================

# Conversion de l'id du fichier clinique en entier pour garantir la correspondance
df_clinique["id"] = df_clinique["id"].astype(int)

# Sélection unique des colonnes nécessaires pour la fusion
df_clinique_sub = df_clinique[["id", "Gender", "Age_at_disease"]]

# Fusion à gauche pour préserver uniquement les lignes validées de df_final
df_final = pd.merge(df_final, df_clinique_sub, on="id", how="left")

# Encodage du Genre (Femme = 0, Homme = 1) en gérant les variations de casse
mapping_genre = {
    "femme": 0,
    "female": 0,
    "f": 0,
    "homme": 1,
    "male": 1,
    "h": 1,
    "m": 1,
}
df_final["Gender"] = (
    df_final["Gender"].astype(str).str.lower().str.strip().map(mapping_genre)
)

# Centrage-réduction des variables cliniques (Gender et Age_at_disease)
for col in ["Gender", "Age_at_disease"]:
    mean_val = df_final[col].mean()
    std_val = df_final[col].std()
    if std_val == 0:
        std_val = 1.0
    df_final[col] = (df_final[col] - mean_val) / std_val

# Définition du nouvel ordre des colonnes de métadonnées
metadata_ordonnee = [
    "id",
    "patient_num",
    "Gender",
    "Age_at_disease",
    "classe_name",
]

# Récupération de toutes les colonnes de features radiomiques (qui excluent les métadonnées)
radiomique_cols = [
    col for col in df_final.columns if col not in metadata_ordonnee
]

# Réassemblage final du DataFrame avec le nouvel ordre de colonnes
df_final = df_final[metadata_ordonnee + radiomique_cols]

df_final.to_csv("./data/flattened_global_scaled3.csv", sep=";", index=False, encoding="utf-8-sig")

In [32]:
import pandas as pd
import numpy as np

def flatten_dataframe(
    df, 
    cols_id,           
    cols_phases,       
    cols_statiques,    
    cols_features      
):
    df = df.copy()
    
    # 0. Normalisation : Si on donne juste un texte, on le transforme en liste
    if isinstance(cols_id, str): cols_id = [cols_id]
    if isinstance(cols_phases, str): cols_phases = [cols_phases]
    
    # 1. Création des clés combinées (Le correctif est ici 🔧)
    # On utilise .apply(lambda row: ...) pour forcer la conversion string ligne par ligne
    target_id = 'combined_id_temp'
    df[target_id] = df[cols_id].apply(lambda row: '_'.join([str(val) for val in row]), axis=1)

    target_phase = 'combined_phase_temp'
    df[target_phase] = df[cols_phases].apply(lambda row: '_'.join([str(val) for val in row]), axis=1)
    
    # 2. Pivot (en utilisant nos clés temporaires)
    df_pivot = df.pivot_table(
        index=target_id, 
        columns=target_phase, 
        values=cols_features,
        aggfunc='first' 
    )
    
    # 3. Renommage des colonnes (ex: 'Volume_T0_Slice1')
    df_pivot.columns = [f"{feature}_{phase}" for feature, phase in df_pivot.columns]
    
    # 4. Préparation des données statiques
    colonnes_a_garder = cols_id + cols_statiques + [target_id]
    df_statique = df[colonnes_a_garder].drop_duplicates(subset=[target_id])
    df_statique = df_statique.set_index(target_id)
    
    # 5. Fusion et Nettoyage
    df_flattened = df_statique.join(df_pivot).reset_index(drop=True)
    
    return df_flattened

In [33]:
df_multi = pd.read_csv("data/multislice_excel_basic(Sheet1).csv", sep=";")
df_flat = flatten_dataframe(
    df_multi, 
    cols_id=["id","temps_inj"], 
    cols_phases=[ "slice_num"], 
    cols_statiques=["classe_name"],
    cols_features=[col for col in df_multi.columns if col.startswith("original_")]
)
display(df_flat.head())

,id,temps_inj,classe_name,original_firstorder_10Percentile_0,original_firstorder_10Percentile_1,original_firstorder_10Percentile_10,original_firstorder_10Percentile_100,original_firstorder_10Percentile_101,original_firstorder_10Percentile_102,original_firstorder_10Percentile_103,...,original_shape2D_Sphericity_90,original_shape2D_Sphericity_91,original_shape2D_Sphericity_92,original_shape2D_Sphericity_93,original_shape2D_Sphericity_94,original_shape2D_Sphericity_95,original_shape2D_Sphericity_96,original_shape2D_Sphericity_97,original_shape2D_Sphericity_98,original_shape2D_Sphericity_99
0,147.0,VEIN,Mixtes,NaN,NaN,NaN,248.8,212.7,194.7,191.0,...,NaN,NaN,NaN,NaN,NaN,NaN,"0,894273121","0,894273121","0,909641206","0,909641206"
1,147.0,TARD,Mixtes,NaN,NaN,NaN,NaN,NaN,NaN,189.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,147.0,PORT,Mixtes,NaN,NaN,NaN,NaN,216.0,215.0,212.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,147.0,ART,Mixtes,NaN,NaN,NaN,260.2,217.8,196.0,190.3,...,NaN,NaN,NaN,"0,87706411","0,87706411","0,859361507","0,829388643","0,829388643","0,853734776","0,853734776"
4,146.0,VEIN,Mixtes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
